# 04 - Hinglish Slur / Abuse Lexicon (Day 9)

Target: ~150 curated base terms, each with romanisation variants.

Workflow:
1. Seed from your existing `hate_lexicon.txt`.
2. Surface candidate terms **from your own hate-labelled posts** (data-driven).
3. You add terms from your native Hindi and mark which candidates to keep.
4. Code auto-generates spelling variants for every kept term.

The variants feed the baseline's lexicon features (Day 10) and the romanisation attack (Objective 4).

### Setup

In [ ]:
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Not on Colab / already mounted:', e)

In [ ]:
import sys; sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')
from hinglish_hate import (load_bohra, load_hasoc2021, load_hasoc2022_threads,
                           build_corpus, filter_romanised, clean_text,
                           generate_variants, variant_column)
import pandas as pd, math, re
from collections import Counter
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/dissertation/data')
def find_one(root,name):
    h=list(Path(root).rglob(name))
    if not h: raise FileNotFoundError(name)
    return h[0]
bohra=load_bohra(find_one(DATA_ROOT,'hate_speech.tsv'))
h21l=list(DATA_ROOT.rglob('labels.json'))
h21=load_hasoc2021(h21l[0].parents[2]) if h21l else None
h22=load_hasoc2022_threads(DATA_ROOT)
roman=filter_romanised(build_corpus([f for f in [bohra,h21,h22] if f is not None]), include_mixed=True)
print('romanised posts:', len(roman))

### 1. Seed from your existing lexicon

In [ ]:
seed=set()
lex=list(DATA_ROOT.rglob('hate_lexicon.txt'))
if lex:
    seed={w.strip().lower() for w in open(lex[0],encoding='utf-8') if w.strip()}
print('seed terms:', len(seed))

### 2. Surface candidates from your hate-labelled data

Scores each token by smoothed log-odds of appearing in a hate post. High score means strongly
associated with the hate class **in your data**. Many will be false positives (topic words, names)
so these are candidates to review, not confirmed terms.

In [ ]:
MIN_COUNT, TOP_N = 5, 150
hate_tok, not_tok = Counter(), Counter()
for t,y in zip(roman['text'], roman['label']):
    toks=set(re.findall(r'[a-z]+', clean_text(t)))
    (hate_tok if y==1 else not_tok).update(toks)
H=int((roman['label']==1).sum()); N=int((roman['label']==0).sum())
rows=[]
for w in set(hate_tok)|set(not_tok):
    ch,cn=hate_tok[w],not_tok[w]
    if ch+cn<MIN_COUNT or len(w)<3: continue
    rows.append((w,ch,cn,round(math.log(((ch+1)/(H+2))/((cn+1)/(N+2))),2), w in seed))
cand=pd.DataFrame(rows,columns=['term','in_hate','in_not','hate_logodds','in_seed'])
cand=cand.sort_values('hate_logodds',ascending=False)
new_cand=cand[~cand['in_seed']].head(TOP_N)
print(f'{len(new_cand)} candidates surfaced. Top 50 for review:')
display(new_cand.head(50).reset_index(drop=True))

### 3. Add your own terms here

**This is the contribution.** Put base terms you know as a native speaker in `MY_TERMS` below,
one per line. Do not worry about spelling variants: the next cell generates those automatically.
Aim for enough here plus kept candidates to reach ~150 total.

Optionally tag categories in `MY_CATEGORIES` (term -> category) for the write-up breakdown.

In [ ]:
MY_TERMS = """
# one base term per line, comments with # are ignored
# e.g.:
# <your term>
"""

MY_CATEGORIES = {
    # 'term': 'caste' | 'religious' | 'gendered' | 'general_abuse' | 'other',
}

my_terms=[l.strip().lower() for l in MY_TERMS.splitlines()
          if l.strip() and not l.strip().startswith('#')]
print(f'{len(my_terms)} terms entered from native knowledge')

### 4. Build the lexicon CSV with auto-generated variants

Every term gets its romanisation variants generated by rule (vowel length, aspiration, k/c/q,
gemination, inflectional endings). Review the `variants` column and prune anything implausible.

In [ ]:
out=[]
for w in sorted(seed):
    out.append({'term':w,'category':MY_CATEGORIES.get(w,''),'keep':1,'source':'seed'})
for w in my_terms:
    out.append({'term':w,'category':MY_CATEGORIES.get(w,''),'keep':1,'source':'native'})
for _,r in new_cand.iterrows():
    if r['term'] in seed or r['term'] in my_terms: continue
    out.append({'term':r['term'],'category':'','keep':'?','source':'data_candidate'})

lex_df=pd.DataFrame(out).drop_duplicates(subset='term').reset_index(drop=True)
lex_df['variants']=variant_column(lex_df['term'], depth=1)
lex_df=lex_df[['term','variants','category','keep','source']]

lex_path=DATA_ROOT/'hinglish_slur_lexicon.csv'
lex_df.to_csv(lex_path,index=False)
print(f'wrote {len(lex_df)} rows -> {lex_path}')
print(lex_df['source'].value_counts().to_dict())
display(lex_df.head(10))

### 5. Curation progress

Run after editing the CSV to see how close you are to ~150 kept terms.

In [ ]:
chk=pd.read_csv(lex_path)
kept=chk[chk['keep'].astype(str).isin(['1','1.0','True','true'])]
n_forms=sum(1+len([v for v in str(r).split('|') if v.strip()]) for r in kept['variants'].fillna(''))
print(f'kept base terms : {len(kept)} / ~150 target')
print(f'total surface forms (terms + variants): {n_forms}')
print(f'undecided candidates left: {(chk["keep"].astype(str)=="?").sum()}')
print('\nby category:'); print(kept['category'].fillna('(blank)').value_counts().to_string())

### Notes for the write-up
- Keep the `source` column: it shows the lexicon is part expert-curated (`native`, `seed`) and part
  data-driven (`data_candidate`), which is a defensible methodology.
- Variants are generated by documented rules (in `variants.py`), not hand-guessed, so the process
  is reproducible and describable in the methodology chapter.
- For provenance, published Hinglish abuse lexicons exist to cross-reference and cite, e.g. the
  HEOT resource in Mathur et al. (2018), 'Did you offend me? Classification of Offensive Tweets in
  Hinglish Language'. Citing an established lexicon alongside your own terms strengthens the
  contribution and gives an external reference point in the viva.